In [2]:
import os, httpx
from dotenv import load_dotenv


load_dotenv()

True

In [2]:


response = httpx.post(
    "https://console.nace.ai/v1/systemone",
    headers={"Authorization": f"Bearer {os.environ['NACE_API_KEY']}"},
    json={
        "model": "drex-v2.0",
        "state": {"a": "Hallo", "b": "test"},
        "questions": {
            "is_urgent": {"type": "noul", "instructions": "Does this convey urgency?"}
        },
    },
)
print(response.json())

{'model': 'drex-v2.0', 'answers': {'is_urgent': {'type': 'noul', 'noul': 0.0335, 'confidence': 0.9665}}, 'usage': {'input_tokens': 21, 'output_tokens': 24}, 'evaluation_time_ms': 46.727, 'request_id': 'req_7857c60118e92d87f795b4221a84bcbc'}


In [3]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

spider_ent_data = get_spider_ent_data(max_items=512)

In [5]:
for input in spider_ent_data[0]["input"]:
    print(len(input["candidate_strings"]))

512
506
504
506
502
505
511
511
363


In [10]:
from types import SimpleNamespace

RULES = ("A column counts as referenced if it appears in SELECT, WHERE, a JOIN condition, GROUP BY, "
         "HAVING, ORDER BY, or inside an aggregation function of the SQL query that answers the question. "
         "If a table is used only via * or COUNT(*) and no other column of it is referenced, "
         "its first column counts as referenced.")

# httpx default timeout is 5s; drex-max with ~500 questions per request needs far longer.
TIMEOUT = httpx.Timeout(600.0, connect=30.0)


def query_drex(state, questions):
    """One drex-max request. questions: {name: instruction}.
    Returns an object with .nouls[name].noul and .usage.input_tokens/.output_tokens
    (same shape as the Jev SDK response, so the Jev eval functions work unchanged)."""
    response = httpx.post(
        "https://console.nace.ai/v1/systemone",
        headers={"Authorization": f"Bearer {os.environ['NACE_API_KEY']}"},
        json={
            "model": "drex-max",
            "state": state,
            "questions": {name: {"type": "noul", "instructions": instr} for name, instr in questions.items()},
        },
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    body = response.json()
    return SimpleNamespace(
        nouls={k: SimpleNamespace(noul=a["noul"]) for k, a in body["answers"].items()},
        usage=SimpleNamespace(input_tokens=body["usage"]["input_tokens"],
                              output_tokens=body["usage"]["output_tokens"]),
    )


def merge_responses(responses):
    # Fuehrt die Responses aller Gruppen einer Frage zu einem Objekt mit .nouls und .usage zusammen.
    return SimpleNamespace(
        nouls={k: a for resp in responses for k, a in resp.nouls.items()},
        usage=SimpleNamespace(
            input_tokens=sum(resp.usage.input_tokens for resp in responses),
            output_tokens=sum(resp.usage.output_tokens for resp in responses),
        ),
    )


def run_drex_schema_linking(data):
    results = []
    for q in data:
        responses = []
        for inp in q["input"]:
            query_drex(
                 state={"question": q["question"], "schema": inp["ddl"], "rules": RULES},
                 questions={c: f"Is column {c} referenced in the SQL query?" for c in inp["candidate_strings"]})
        results.append({"question": q["question"], "db_id": q["db"], "sql": q["sql"],
                        "response": merge_responses(responses), "gold_schema": q["gold_schema"]})
    return results

In [7]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

results = run_drex_schema_linking(get_spider_ent_data(max_items=512)[:1])
print(results)

[{'question': 'What are the names and release years for all the songs of the youngest singer?', 'db_id': 'arts_culture_and_media', 'sql': 'SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1', 'response': namespace(nouls={}, usage=namespace(input_tokens=0, output_tokens=0)), 'gold_schema': {'zentra11_concer_artst_profile': ['artist_rep_title', 'art_rep_rel_cal_year', 'artist_age_at_data_entry']}}]


In [9]:
# Zweistufiges Schema Linking mit drex-max (analog zu jev_test.ipynb):
#   Stufe 1: kompaktes Schema (Tabellen, Spalten, FK-Ziele) -> ein Noul pro Tabelle
#   Stufe 2: volle DDL nur der ausgewaehlten Tabellen -> ein Noul pro Spalte
import random

from src.utils import _extract_fk_targets, _pack_tables_into_chunks

TABLE_RULES = ("A table is needed if at least one of its columns is referenced in the SQL query that answers the question, "
               "or if it is needed to join other needed tables. "
               "The schema shown is only one part of a large database; the needed tables may be in another part.")


def compact_table(t):
    # 'tabelle(spalte1, spalte2) FK -> ziel1, ziel2' - ohne Typen, Backticks und Beispielwerte
    line = f"{t['candidates']['table']}({', '.join(t['candidates']['columns'])})"
    fks = _extract_fk_targets(t['ddl'])
    if fks:
        line += f" FK -> {', '.join(sorted(fks))}"
    return line


def chunk_compact(tables, max_tables):
    # Greedy wie _pack_tables_into_chunks, aber nach Zeichen auf dem kompakten Format
    chunks, current = [], []
    for t in tables:
        line = compact_table(t)
        if current and len(current) >= max_tables:
            chunks.append(current)
            current = []
        current.append((t['candidates']['table'], line))
    if current:
        chunks.append(current)
    return chunks


def fk_neighbors(tables, selected):
    # Tabellen, die per FK direkt mit einer ausgewaehlten Tabelle verbunden sind (beide Richtungen)
    neighbors = set()
    for t in tables:
        name = t['candidates']['table'].lower()
        targets = {x.lower() for x in _extract_fk_targets(t['ddl'])}
        if name in selected:
            neighbors |= targets
        if targets & selected:
            neighbors.add(name)
    return neighbors - selected



def _ask(q, state, questions, responses):
    try:
        responses.append(query_drex(state=state, questions=questions))
    except Exception as e:
        print(f"Error processing question {q['question']} with db_id {q['db']}: {e}")

def run_drex_two_stage(data, table_threshold=0.3, include_fk_neighbors=False):
    results = []
    for q in data:
        # Stufe 1: Tabellen
        table_responses = []
        for chunk in chunk_compact(q['db_schema'], 512):
            _ask(q,
                 state={"question": q['question'], "schema": "\n".join(line for _, line in chunk), "rules": TABLE_RULES},
                 questions={name: f"Is table {name} needed for the SQL query?" for name, _ in chunk},
                 responses=table_responses)
        table_scores = {k: a.noul for resp in table_responses for k, a in resp.nouls.items()}

        selected = {t.lower() for t, p in table_scores.items() if p >= table_threshold}
        if include_fk_neighbors:
            selected |= fk_neighbors(q['db_schema'], selected)
        selected_tables = [t for t in q['db_schema'] if t['candidates']['table'].lower() in selected]

        # Stufe 2: Spalten der ausgewaehlten Tabellen
        column_responses = []
        for chunk in _pack_tables_into_chunks(selected_tables, q['question'], None, q['db'], None, None, 512):
            _ask(q,
                 state={"question": q['question'], "schema": chunk['ddl'], "rules": RULES},
                 questions={c: f"Is column {c} referenced in the SQL query?" for c in chunk['candidate_strings']},
                 responses=column_responses)

        results.append({"question": q['question'], "db_id": q['db'], "sql": q['sql'], "gold_schema": q['gold_schema'],
                        "table_scores": table_scores, "selected_tables": sorted(selected),
                        "table_response": merge_responses(table_responses),
                        "response": merge_responses(column_responses)})
    return results


def stage1_table_recall(results, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    # Recall auf Tabellenebene: Wie viele Gold-Tabellen haette Stufe 1 bei welcher Schwelle durchgelassen?
    print(f"{'thr':>5} {'R_tab':>7} {'R=1':>7} {'#tab':>7}")
    for thr in thresholds:
        hit = total = n_sel = all_found = 0
        for r in results:
            gold = {t.lower() for t in r['gold_schema'] if t}
            sel = {t.lower() for t, p in r['table_scores'].items() if p >= thr}
            hit += len(gold & sel)
            total += len(gold)
            n_sel += len(sel)
            all_found += gold <= sel
        print(f"{thr:>5.2f} {hit / total:>7.4f} {all_found / len(results):>7.4f} {n_sel / len(results):>7.2f}")

In [33]:
table_chunks = chunk_compact(spider_ent_data[129]["db_schema"], 512)

for input in table_chunks:
    print(len(input))

512
27


In [15]:
# Eval: micro P/R/F6 ueber (table, column)-Paare, identisch zu jev_test.ipynb / src/QLora/evaluate.py
import json


def response_to_dict(resp):
    return {
        "nouls": {k: a.noul for k, a in resp.nouls.items()},
        "usage": {"input_tokens": resp.usage.input_tokens,
                  "output_tokens": resp.usage.output_tokens},
    }


def save_results(results, file_name):
    serializable = []
    for r in results:
        item = {**r, "response": response_to_dict(r["response"])}
        # Nur bei run_drex_two_stage vorhanden: zusammengefuehrte Responses aus Stufe 1 (Tabellen)
        if "table_response" in r:
            item["table_response"] = response_to_dict(r["table_response"])
        serializable.append(item)
    with open(f"{file_name}.json", "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=4)


def _gold_cols(gold):
    return {(t.lower(), c.lower()) for t, cols in gold.items() for c in (cols or []) if t and c}

def _scores(response):
    # 'table.column' -> noul-Wahrscheinlichkeit
    return {tuple(k.lower().split(".", 1)): a.noul for k, a in response.nouls.items()}

def eval_results(results, threshold=0.5, beta=6):
    tp = n_pred = n_gold = 0
    perfect_recall = 0
    for r in results:
        gold = _gold_cols(r["gold_schema"])
        pred = {k for k, p in _scores(r["response"]).items() if p >= threshold}
        tp += len(pred & gold)
        n_pred += len(pred)
        n_gold += len(gold)
        perfect_recall += gold <= pred
    p = tp / n_pred if n_pred else 0.0
    rec = tp / n_gold if n_gold else 0.0
    b2 = beta ** 2
    f = (1 + b2) * p * rec / (b2 * p + rec) if (b2 * p + rec) else 0.0
    return {"threshold": threshold, "precision": p, "recall": rec, f"f{beta}": f,
            "perfect_recall_rate": perfect_recall / len(results),
            "avg_pred_cols": n_pred / len(results), "tp": tp, "pred": n_pred, "gold": n_gold}

def present_results(results):
    print(f"Fragen: {len(results)}")

    # Gold-Spalten, die gar nicht unter den Kandidaten sind
    missing = sum(len(_gold_cols(r["gold_schema"]) - set(_scores(r["response"]))) for r in results)
    print(f"Gold-Spalten ohne Kandidat: {missing}")

    usage_in = sum(r["response"].usage.input_tokens for r in results)
    usage_out = sum(r["response"].usage.output_tokens for r in results)
    print(f"Tokens: input={usage_in:,}  output={usage_out:,}\n")

    print(f"{'thr':>5} {'P':>7} {'R':>7} {'F6':>7} {'R=1':>7} {'#pred':>7}")
    for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        m = eval_results(results, thr)
        print(f"{thr:>5.2f} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f6']:>7.4f} "
            f"{m['perfect_recall_rate']:>7.4f} {m['avg_pred_cols']:>7.2f}")

    # Fehleranalyse bei 0.5: Fragen mit verpassten Gold-Spalten
    misses = []
    for r in results:
        s = _scores(r["response"])
        missed = {k: s.get(k) for k in _gold_cols(r["gold_schema"]) if s.get(k, 0) < 0.5}
        if missed:
            misses.append((r["question"], r["sql"], missed))
    print(f"\nFragen mit verpassten Gold-Spalten @0.5: {len(misses)}")
    for qtext, sql, missed in misses[:10]:
        print(f"- {qtext}\n  {sql}\n  missed: {missed}")

In [11]:
results_two_stage = run_drex_two_stage(spider_ent_data[:1], table_threshold=0.3, include_fk_neighbors=False)


In [12]:
results_two_stage

[{'question': 'What are the names and release years for all the songs of the youngest singer?',
  'db_id': 'arts_culture_and_media',
  'sql': 'SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1',
  'gold_schema': {'zentra11_concer_artst_profile': ['artist_rep_title',
    'art_rep_rel_cal_year',
    'artist_age_at_data_entry']},
  'table_scores': {'xelion42_bridom_architect': 0.0001606487276327102,
   'xelion42_bridom_brg_structure': 0.00013340245544433383,
   'xelion42_bridom_millreg': 0.00016064858213692877,
   'zynox_artwk_artist': 0.0009679732089613759,
   'zynox_artwk_pntngs': 0.000803915121532267,
   'zynox_artwk_sculptures': 0.000803914307280152,
   'xylon_bcast_channel': 0.0002984688241556548,
   'xylon_bcast_dir_info': 0.0009679722203889719,
   'xylon_bcast_prgm_overview': 0.0009098722842210736,
   'xylon_bcast_dir_admin': 0.00015099874012531052,
   'xanfor_bksys_client': 0.00017091514548342595,
   'xanfor_bksys_order_txn': 0.00021897408157120433,
   'xanfor

In [ ]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

# Gleiche 30 Fragen wie im Jev-Test (seed 42) -> direkt vergleichbar
spider_ent = get_spider_ent_data(max_items=512)
sample = random.Random(42).sample(spider_ent, 30)

results_two_stage = run_drex_two_stage(sample, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)

save_results(results_two_stage, "spider_ent_drex_max_two_stage_results_30_samples_seed_42")

KeyboardInterrupt: 

In [13]:
results_two_stage = run_drex_two_stage(spider_ent_data, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)

save_results(results_two_stage, "spider_ent_drex_max_two_stage_results")

Stufe 1 (Tabellen):
  thr   R_tab     R=1    #tab
 0.10  0.9886  0.9834    2.69
 0.20  0.9829  0.9751    2.28
 0.30  0.9795  0.9701    2.11
 0.40  0.9772  0.9668    2.02
 0.50  0.9749  0.9635    1.93
Tokens Stufe 1: input=38,378,842

Stufe 2 (Spalten):


NameError: name 'present_results' is not defined

In [14]:
results_two_stage

[{'question': 'What are the names and release years for all the songs of the youngest singer?',
  'db_id': 'arts_culture_and_media',
  'sql': 'SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1',
  'gold_schema': {'zentra11_concer_artst_profile': ['artist_rep_title',
    'art_rep_rel_cal_year',
    'artist_age_at_data_entry']},
  'table_scores': {'xelion42_bridom_architect': 0.0001606487276327102,
   'xelion42_bridom_brg_structure': 0.00013340245544433383,
   'xelion42_bridom_millreg': 0.00016064858213692877,
   'zynox_artwk_artist': 0.0009679732089613759,
   'zynox_artwk_pntngs': 0.000803915121532267,
   'zynox_artwk_sculptures': 0.000803914307280152,
   'xylon_bcast_channel': 0.0002984688241556548,
   'xylon_bcast_dir_info': 0.0009679722203889719,
   'xylon_bcast_prgm_overview': 0.0009098722842210736,
   'xylon_bcast_dir_admin': 0.00015099874012531052,
   'xanfor_bksys_client': 0.00017091514548342595,
   'xanfor_bksys_order_txn': 0.00021897408157120433,
   'xanfor

In [16]:
# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)

save_results(results_two_stage, "spider_ent_drex_max_two_stage_results")

Stufe 2 (Spalten):
Fragen: 602
Gold-Spalten ohne Kandidat: 33
Tokens: input=1,252,935  output=468,017

  thr       P       R      F6     R=1   #pred
 0.10  0.6783  0.9698  0.9586  0.9468    3.85
 0.20  0.7093  0.9685  0.9590  0.9452    3.67
 0.30  0.7342  0.9667  0.9585  0.9419    3.54
 0.40  0.7533  0.9630  0.9558  0.9319    3.44
 0.50  0.7684  0.9605  0.9540  0.9269    3.36
 0.60  0.7969  0.9543  0.9493  0.9103    3.22
 0.70  0.8256  0.9494  0.9455  0.9003    3.09
 0.80  0.8566  0.9364  0.9341  0.8721    2.94
 0.90  0.8905  0.9086  0.9081  0.8239    2.75

Fragen mit verpassten Gold-Spalten @0.5: 44
- What are the names and release years for all the songs of the youngest singer?
  SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
  missed: {('zentra11_concer_artst_profile', 'artist_age_at_data_entry'): None, ('zentra11_concer_artst_profile', 'artist_rep_title'): None, ('zentra11_concer_artst_profile', 'art_rep_rel_cal_year'): None}
- What is the first name of ever